# VOICE_CLONE_TTS — Nhân bản giọng nói (Text-to-Speech) — Bản nguồn mở
## Cách dùng nhanh: bấm "Chạy tất cả" (Run all), đợi khoảng 5 phút cài đặt là dùng được.

VUI LÒNG ĐỌC HẾT HƯỚNG DẪN BÊN DƯỚI TRƯỚC KHI SỬ DỤNG.

---

## GIỚI THIỆU
* Bản xây dựng sạch từ mã nguồn mở **F5-TTS** (https://github.com/SWivid/F5-TTS), minh bạch, không dùng app mã hóa.
* Chức năng: **nhân bản giọng nói** — tải lên 1 file giọng mẫu (5–12 giây), nhập văn bản bất kỳ, máy sẽ đọc bằng đúng giọng đó.
* Hỗ trợ **KHÔNG GIỚI HẠN KÍ TỰ** (tự động chia nhỏ văn bản dài).
* **Đã tối ưu chống thiếu chữ / vấp**: văn bản được chia theo câu với độ dài cố định, mỗi đoạn được kết thúc đầy đủ dấu câu trước khi ghép nối.
* Model mặc định: **F5-TTS v1 Base (Anh + Trung Quốc)**. Giao diện có sẵn hơn **10 model ngôn ngữ**: tiếng Việt, Anh, Trung, Nhật, Pháp, Đức, Ý, Tây Ban Nha, Nga, Phần Lan, Latvia, Hindi...

## 👉 HƯỚNG DẪN:
1. Trên trình đơn Colab: **Runtime -> Change runtime type -> T4 GPU** (cực kì quan trọng).
2. **Bấm "Chạy tất cả"** để cài đặt và khởi động.
3. Lần chạy đầu tiên sẽ tải model (~1.3 GB) và mất vài phút.
4. Sau khi có link giao diện (dòng cuối cùng của TASK 6), mở trong tab mới.
5. Tải lên **file giọng mẫu** (WAV/MP3 sạch, không nhạc nền, 5–12 giây), nhập văn bản, bấm **Synthesize**.
6. Chọn **model theo ngôn ngữ** cần đọc (vd. `Tiếng Việt`, `Tiếng Nhật`, `Tiếng Pháp`...). Lần đầu chọn model nào máy sẽ tải model đó (~1.3 GB).
7. **Lưu giọng để dùng lại**: điền Tên giọng rồi bấm `Lưu giọng mẫu này` — lần sau chỉ cần chọn tên trong danh sách `Giọng đã lưu` là có lại đúng file giọng + nội dung cũ.
8. (Tùy chọn) Chạy cell **TASK 6 — GẮN GOOGLE DRIVE** để giọng đã lưu được giữ lâu dài giữa các phiên Colab.

## NẾU VẪN CÒN THIẾU CHỮ / VẤP:
* Giảm thanh trượt **"Độ dài đoạn (max_chars)"** xuống 60–80.
* Nhập chính xác **"Nội dung trong file giọng mẫu"** (không để trống) để máy tính đúng độ dài.
* Dùng file giọng mẫu ngắn, rõ (5–10 giây), không nhạc nền.
* Tăng **nfe_step** lên 48–64 nếu bị rè hoặc vấp.

## NẾU KẾT QUẢ NÓI THỪA / LẶP LẠI 1 ĐOẠN VĂN BẢN:
* Nguyên nhân chính: model nhồi "Nội dung trong file giọng mẫu" (ref_text) vào chung chuỗi sinh audio, khi độ dài giọng mẫu bị sai/lệch thì **đuôi của ref_text bị đọc thêm ra**. Đoạn thừa đó chính là văn bản bạn đã nhập cho giọng mẫu lúc trước.
* Máy sẽ **cảnh báo** nếu giọng mẫu dài hơn 12 giây (lúc đó F5-TTS tự cắt cụt giọng mẫu nhưng giữ nguyên nội dung — dễ gây thừa/lặp). Nếu vẫn còn thừa:
* Dùng file giọng mẫu **dưới 12 giây**, cắt gọn (đầu/cuối có chút lặng), không nhạc nền.
* Nếu không chắc "Nội dung trong file giọng mẫu" khớp 100%, hãy **để trống** để máy tự nhận dạng đúng độ dài.
* Tăng **nfe_step** lên 48–64, hoặc tăng nhẹ **cfg_strength** lên 2.5–3.0 để giảm ảo giác lặp chữ.

## ⚠️ LƯU Ý:
* Chỉ dùng cho mục đích cá nhân / học tập. **Không** nhân bản giọng nói của người khác khi chưa được phép.
* Model F5-TTS dùng giấy phép **CC-BY-NC** (phi thương mại).


In [ ]:
# @title TASK 1 — KIỂM TRA MÔI TRƯỜNG (GPU / FFMPEG)
import subprocess, sys

# 1) Kiểm tra GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: Không có GPU. Vào Runtime -> Change runtime type -> chọn T4 GPU rồi chạy lại.")

# 2) Cài ffmpeg nếu chưa có (cần để đọc/xuất âm thanh)
try:
    subprocess.run(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    print("ffmpeg: OK")
except Exception:
    print("ffmpeg: đang cài đặt...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)
    print("ffmpeg: xong")


In [ ]:
# @title TASK 2 — CÀI ĐẶT F5-TTS
# Trên Colab, torch/torchaudio đã có sẵn. pip install f5-tts sẽ bổ sung các thư viện còn thiếu.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "f5-tts"], check=True)
print("Đã cài xong F5-TTS.")


In [ ]:
# @title TASK 3 — TẢI & NẠP MODEL (nhiều ngôn ngữ)
import os
from f5_tts.api import F5TTS
from huggingface_hub import hf_hub_download

# Danh sách model ngôn ngữ có sẵn. Model được tải "lười" (chỉ tải khi chọn trong giao diện).
MODELS = {
    "🇻🇳 Tiếng Việt (F5-TTS-Vietnamese)": {"model": "F5TTS_Base", "repo_id": "toandev/F5-TTS-Vietnamese", "subfolder": None, "ckpt_name": "model_latest.safetensors", "vocab_name": "vocab.txt"},
    "🇬🇧🇨🇳 F5-TTS v1 Base (Anh + Trung)": {"model": "F5TTS_v1_Base", "repo_id": None, "subfolder": None, "ckpt_name": None, "vocab_name": None},
    "🇬🇧🇨🇳 F5-TTS Base (Anh + Trung)": {"model": "F5TTS_Base", "repo_id": None, "subfolder": None, "ckpt_name": None, "vocab_name": None},
    "🇯🇵 Tiếng Nhật (Jmica/F5TTS)": {"model": "F5TTS_Base", "repo_id": "Jmica/F5TTS", "subfolder": "JA_21999120", "ckpt_name": "model_21999120.pt", "vocab_name": "vocab_japanese.txt"},
    "🇫🇷 Tiếng Pháp (RASPIAUDIO/F5-French)": {"model": "F5TTS_Base", "repo_id": "RASPIAUDIO/F5-French-MixedSpeakers-reduced", "subfolder": None, "ckpt_name": "model_last_reduced.pt", "vocab_name": "vocab.txt"},
    "🇩🇪 Tiếng Đức (hvoss-techfak/F5-TTS-German)": {"model": "F5TTS_Base", "repo_id": "hvoss-techfak/F5-TTS-German", "subfolder": None, "ckpt_name": "model_f5tts_german.pt", "vocab_name": "vocab.txt"},
    "🇮🇹 Tiếng Ý (alien79/F5-TTS-italian)": {"model": "F5TTS_Base", "repo_id": "alien79/F5-TTS-italian", "subfolder": None, "ckpt_name": "model_159600.safetensors", "vocab_name": "vocab.txt"},
    "🇪🇸 Tiếng Tây Ban Nha (jpgallegoar/F5-Spanish)": {"model": "F5TTS_Base", "repo_id": "jpgallegoar/F5-Spanish", "subfolder": None, "ckpt_name": "model_last.pt", "vocab_name": "vocab.txt"},
    "🇷🇺 Tiếng Nga (hotstone228/F5-TTS-Russian)": {"model": "F5TTS_Base", "repo_id": "hotstone228/F5-TTS-Russian", "subfolder": None, "ckpt_name": "model_last.safetensors", "vocab_name": "vocab.txt"},
    "🇫🇮 Tiếng Phần Lan (AsmoKoskinen/F5-TTS-Finnish)": {"model": "F5TTS_Base", "repo_id": "AsmoKoskinen/F5-TTS_Finnish_Model", "subfolder": None, "ckpt_name": "model_common_voice_fi_vox_populi_fi_20241206.safetensors", "vocab_name": "vocab.txt"},
    "🇱🇻 Tiếng Latvia (RaivisDejus/F5-TTS-Latvian)": {"model": "F5TTS_Base", "repo_id": "RaivisDejus/F5-TTS-Latvian", "subfolder": None, "ckpt_name": "model.safetensors", "vocab_name": "vocab.txt"},
    "🇮🇳 Tiếng Hindi (SPRINGLab/F5-Hindi-24KHz)": {"model": "F5TTS_small", "repo_id": "SPRINGLab/F5-Hindi-24KHz", "subfolder": None, "ckpt_name": "model_2500000.safetensors", "vocab_name": "vocab.txt"},
}

ENGINES = {}  # cache các engine đã nạp (dùng chung cho các task sau)

def load_engine(model_key):
    """Nạp (hoặc lấy lại nếu đã nạp) engine cho một model_key trong MODELS."""
    if model_key in ENGINES:
        return ENGINES[model_key]
    info = MODELS[model_key]
    print(f"Đang tải và nạp model: {model_key} ...")
    if info["repo_id"]:
        ckpt = hf_hub_download(info["repo_id"], info["ckpt_name"], subfolder=info["subfolder"])
        vocab = hf_hub_download(info["repo_id"], info["vocab_name"], subfolder=info["subfolder"])
        ENGINES[model_key] = F5TTS(model=info["model"], ckpt_file=ckpt, vocab_file=vocab)
    else:
        ENGINES[model_key] = F5TTS(model=info["model"])
    return ENGINES[model_key]

VIETNAMESE_MODEL = False  # True = nạp sẵn model tiếng Việt ngay từ đầu
DEFAULT_MODEL = "🇻🇳 Tiếng Việt (F5-TTS-Vietnamese)" if VIETNAMESE_MODEL else "🇬🇧🇨🇳 F5-TTS v1 Base (Anh + Trung)"
engine = load_engine(DEFAULT_MODEL)
print("OK — model đã sẵn sàng:", DEFAULT_MODEL)


In [ ]:
# @title TASK 4 — HÀM SINH ỔN ĐỊNH (chống thiếu chữ / vấp)
# Lý do gốc của lỗi "thiếu chữ" và "vấp": F5-TTS chia văn bản thành các đoạn
# (chunk) có độ dài tự động có thể rất lớn, và đoạn cuối mỗi chunk hay bị cắt mất từ.
# Hàm dưới đây ép chia theo CÂU với độ dài CỐ ĐỊNH, và thêm dấu câu đầy đủ cho mỗi đoạn.
import os, random, sys
import numpy as np
import torch
import torchaudio
import soundfile as sf
from f5_tts.infer.utils_infer import (
    preprocess_ref_audio_text,
    chunk_text,
    infer_batch_process,
    target_sample_rate,
    target_rms,
    cross_fade_duration,
    remove_silence_for_generated_wav,
)
from f5_tts.model.utils import seed_everything


def _split_clean(text, max_chars):
    """Chia văn bản theo câu, giới hạn mỗi đoạn ~max_chars ký tự, luôn kết thúc bằng dấu câu."""
    batches = chunk_text(text, max_chars=int(max_chars))
    out = []
    for b in batches:
        b = b.strip()
        if not b:
            continue
        if b[-1] not in ".!?。！？":
            b = b + "."
        out.append(b)
    return out


def prepare_ref(ref_file, ref_text):
    """Chống lỗi \"nói thừa / lặp lại 1 đoạn văn bản\": chỉ KIỂM TRA, không sửa file.
    Nếu giọng mẫu dài hơn 12s, F5-TTS sẽ tự cắt cụt audio lúc preprocess nhưng giữ nguyên
    ref_text -> audio lệch text -> model đọc thêm/lặp lại cụm cuối của ref_text.
    Ở đây chỉ in cảnh báo để bạn tự xử lý (an toàn, không đổi hành vi so với bản gốc)."""
    try:
        wav, sr = torchaudio.load(ref_file)
        duration = wav.shape[1] / sr
        if duration > 12.0:
            print(f"⚠️ Giọng mẫu dài {duration:.1f}s (> 12s): F5-TTS sẽ cắt cụt giọng mẫu. "
                  f"Nên dùng giọng mẫu < 12s hoặc để trống 'Nội dung trong file giọng mẫu' "
                  f"để tránh nói thừa/lặp 1 đoạn.")
    except Exception:
        pass
    return ref_file, ref_text or ""


def stable_generate(engine, ref_file, ref_text, gen_text, max_chars=100,
                    speed=1.0, nfe=32, cfg=2.0, seed=None):
    """Sinh giọng ổn định: trả về (wave, sr, spec, seed_da_dung)."""
    if seed is None:
        seed = random.randint(0, sys.maxsize)
    seed_everything(seed)

    ref_audio, ref_text = prepare_ref(ref_file, ref_text or "")
    ref_audio, ref_text = preprocess_ref_audio_text(ref_audio, ref_text, show_info=print)
    batches = _split_clean(gen_text, max_chars=max_chars)
    print("Số đoạn cần sinh:", len(batches))
    if not batches:
        return None, target_sample_rate, None, seed

    audio, sr = torchaudio.load(ref_audio)
    gen = infer_batch_process(
        (audio, sr),
        ref_text,
        batches,
        engine.ema_model,
        engine.vocoder,
        mel_spec_type=engine.mel_spec_type,
        progress=None,
        target_rms=target_rms,
        cross_fade_duration=cross_fade_duration,
        nfe_step=int(nfe),
        cfg_strength=float(cfg),
        sway_sampling_coef=-1.0,
        speed=speed,
        fix_duration=None,
        device=engine.device,
    )
    try:
        wave, s, spec = next(gen)
    except StopIteration:
        wave, s, spec = None, target_sample_rate, None
    return wave, s, spec, seed


def save_wav(wave, sr, path, remove_silence=False):
    sf.write(path, wave, sr)
    if remove_silence:
        remove_silence_for_generated_wav(path)
    return path


In [ ]:
# @title TASK 5 — CHẠY THỬ (dùng giọng mẫu có sẵn)
import os
from importlib.resources import files

os.makedirs("/content/output", exist_ok=True)

ref_audio = str(files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
ref_text = "Some call me nature, others call me mother nature."
gen_text = "Hello! This is a quick test of your cloned voice. Keep this file as your reference voice sample. " \
           "The text is split into small complete sentences so that every single word is pronounced fully and clearly."

wave, sr, spec, seed = stable_generate(engine, ref_audio, ref_text, gen_text,
                                       max_chars=100, speed=1.0, nfe=32, cfg=2.0)
out_path = save_wav(wave, sr, "/content/output/test_output.wav")
print("Đã tạo:", out_path, "| seed =", seed, "| sr =", sr)

from IPython.display import Audio
Audio(out_path)


In [ ]:
# @title TASK 6 — GẮN GOOGLE DRIVE (tùy chọn, để giọng lưu được lâu dài)
# KHÔNG bắt buộc. Chạy cell này nếu muốn các giọng đã lưu được giữ lại giữa các phiên Colab.
# (Không gắn Drive thì giọng vẫn lưu được trong phiên hiện tại, nhưng mất khi đóng runtime.)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Đã gắn Google Drive. Giọng đã lưu sẽ được lưu vào MyDrive/VOICE_CLONE_TTS/voices/")
except Exception as e:
    print("Bỏ qua gắn Drive (không sao):", e)


In [ ]:
# @title TASK 7 — THƯ VIỆN GIỌNG ĐÃ LƯU (lưu / nạp lại giọng mẫu đã dùng)
# Lưu nhiều giọng mẫu đã dùng (kèm nội dung) để dùng lại nhanh. Nếu đã gắn Google Drive
# (cell TASK 6) giọng sẽ được giữ lâu dài; nếu không, giọng lưu trong phiên hiện tại.
import os, json, shutil, re

VOICE_DIR = "/content/voices"
DRIVE_VOICE_DIR = "/content/drive/MyDrive/VOICE_CLONE_TTS/voices"

def _voices_root():
    if os.path.isdir("/content/drive/MyDrive"):
        return DRIVE_VOICE_DIR
    return VOICE_DIR

def _load_index():
    index_path = os.path.join(_voices_root(), "voices.json")
    if os.path.exists(index_path):
        with open(index_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def _save_index(index):
    root = _voices_root()
    os.makedirs(root, exist_ok=True)
    with open(os.path.join(root, "voices.json"), "w", encoding="utf-8") as f:
        json.dump(index, f, ensure_ascii=False, indent=2)

def _safe_name(name):
    name = re.sub(r"[^\w\- ]", "", name).strip()
    return name or "voice"

def list_voices():
    return list(_load_index().keys())

def save_voice(name, audio_path, ref_text):
    """Lưu giọng mẫu hiện tại vào thư viện. Trả về (danh sách giọng, thông báo)."""
    if not (name or "").strip():
        return list_voices(), "⚠️ Chưa nhập Tên giọng để lưu."
    if not audio_path:
        return list_voices(), "⚠️ Chưa có file giọng mẫu (ref audio) để lưu."
    root = _voices_root()
    os.makedirs(root, exist_ok=True)
    name = _safe_name(name)
    ext = os.path.splitext(audio_path)[1] or ".wav"
    dest = os.path.join(root, name + ext)
    shutil.copyfile(audio_path, dest)
    index = _load_index()
    index[name] = {"audio": dest, "ref_text": ref_text or ""}
    _save_index(index)
    return list_voices(), f"✅ Đã lưu giọng \"{name}\" — chọn lại trong danh sách là dùng được."

def load_voice(name):
    """Nạp giọng đã lưu (trả về path audio + ref_text) để gán vào giao diện."""
    index = _load_index()
    if name and name in index:
        return index[name]["audio"], index[name].get("ref_text", "")
    return None, None

def delete_voice(name):
    """Xóa giọng đã lưu. Trả về (danh sách giọng, thông báo)."""
    index = _load_index()
    if name and name in index:
        audio = index[name].get("audio")
        if audio and os.path.exists(audio):
            os.remove(audio)
        del index[name]
        _save_index(index)
        return list_voices(), f"🗑 Đã xóa giọng \"{name}\"."
    return list_voices(), "Không tìm thấy giọng đã chọn."

print("Số giọng đã lưu hiện có:", len(list_voices()))


In [ ]:
# @title TASK 8 — KHỞI ĐỘNG GIAO DIỆN NHÂN BẢN GIỌNG NÓI
import os
import gradio as gr

os.makedirs("/content/output", exist_ok=True)

def synthesize(model_key, ref_audio, ref_text, gen_text, speed, nfe, cfg, max_chars, remove_silence, seed_text):
    if ref_audio is None:
        raise gr.Error("Vui lòng tải lên file giọng mẫu (ref audio).")
    if not (gen_text or "").strip():
        raise gr.Error("Vui lòng nhập văn bản cần đọc.")
    eng = load_engine(model_key)
    seed = None if not (seed_text or "").strip() else int(seed_text)
    wave, sr, spec, seed_used = stable_generate(
        eng, ref_audio, (ref_text or ""), gen_text,
        max_chars=max_chars, speed=speed, nfe=nfe, cfg=cfg, seed=seed,
    )
    if wave is None:
        raise gr.Error("Không sinh được âm thanh. Kiểm tra lại file giọng mẫu / văn bản.")
    out = save_wav(wave, sr, "/content/output/cloned_output.wav", remove_silence=remove_silence)
    return out, f"Xong. seed = {seed_used} | {len(gen_text)} ký tự"

def _dropdown_update(choices, value=None):
    """Cập nhật Dropdown, tương thích nhiều phiên bản Gradio."""
    try:
        return gr.update(choices=choices, value=value)
    except (AttributeError, TypeError):
        return gr.Dropdown(choices=choices, value=value)

def load_voice_to_ui(name):
    audio, text = load_voice(name)
    if audio:
        return audio, text or "", f"✅ Đã nạp giọng \"{name}\". Bấm Synthesize là dùng được."
    return None, "", "Không tìm thấy giọng đã chọn."

def refresh_voices():
    return _dropdown_update(list_voices())

def save_voice_ui(name, audio_path, ref_text):
    names, msg = save_voice(name, audio_path, ref_text)
    return _dropdown_update(names), msg

def delete_voice_ui(name):
    names, msg = delete_voice(name)
    return _dropdown_update(names), msg

with gr.Blocks(title="VOICE_CLONE_TTS") as demo:
    gr.Markdown("### Nhân bản giọng nói (F5-TTS) — đã tối ưu chống thiếu chữ / vấp")
    gr.Markdown("#### Bước 1 — chọn model ngôn ngữ, tải file giọng mẫu (hoặc chọn giọng đã lưu)")
    with gr.Row():
        with gr.Column():
            model_key = gr.Dropdown(list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model ngôn ngữ (lần đầu chọn model nào sẽ tải model đó, mất vài phút)")
            ref_audio = gr.Audio(type="filepath", label="File giọng mẫu (5-12 giây, sạch)")
            ref_text = gr.Textbox(label="Nội dung trong file giọng mẫu (để trống = tự nhận dạng; nhập sai/khớp không đúng sẽ dễ bị nói thừa 1 đoạn)", placeholder="Không bắt buộc, nhưng nhập chính xác thì đỡ thiếu chữ hơn")
            gen_text = gr.Textbox(label="Văn bản cần đọc (không giới hạn ký tự)", lines=6)
            with gr.Accordion("Tùy chọn nâng cao", open=False):
                max_chars = gr.Slider(40, 250, value=100, step=5, label="Độ dài đoạn (max_chars) — càng nhỏ càng ít thiếu chữ nhưng dễ vấp hơn")
                speed = gr.Slider(0.3, 2.0, value=1.0, step=0.05, label="Tốc độ đọc")
                nfe = gr.Slider(8, 64, value=32, step=1, label="Số bước khử nhiễu (nfe_step)")
                cfg = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label="Cường độ bám giọng (cfg_strength)")
                seed_text = gr.Textbox(label="Seed (để trống = ngẫu nhiên)", placeholder="Không bắt buộc")
                remove_silence = gr.Checkbox(value=True, label="Loại bỏ khoảng lặng thừa")
        with gr.Column():
            out_audio = gr.Audio(type="filepath", label="Kết quả")
            info = gr.Textbox(label="Trạng thái", interactive=False)
    btn = gr.Button("Synthesize", variant="primary")
    btn.click(synthesize, inputs=[model_key, ref_audio, ref_text, gen_text, speed, nfe, cfg, max_chars, remove_silence, seed_text], outputs=[out_audio, info])

    gr.Markdown("#### Bước 2 — thư viện giọng: lưu giọng mẫu đã dùng để dùng lại lần sau")
    with gr.Row():
        voice_name = gr.Textbox(label="Tên giọng mới", placeholder="VD: Giọng Nam, Giọng chị Lan...")
        saved_voices = gr.Dropdown(choices=list_voices(), label="Giọng đã lưu", interactive=True)
    with gr.Row():
        btn_save = gr.Button("💾 Lưu giọng mẫu này")
        btn_load = gr.Button("📂 Nạp giọng đã chọn")
        btn_delete = gr.Button("🗑 Xóa giọng đã chọn")
    voice_msg = gr.Textbox(label="Tin nhắn thư viện giọng", interactive=False)

    btn_save.click(save_voice_ui, inputs=[voice_name, ref_audio, ref_text], outputs=[saved_voices, voice_msg])
    btn_load.click(load_voice_to_ui, inputs=[saved_voices], outputs=[ref_audio, ref_text, voice_msg])
    btn_delete.click(delete_voice_ui, inputs=[saved_voices], outputs=[saved_voices, voice_msg])
    demo.load(refresh_voices, outputs=[saved_voices])

try:
    demo.launch(share=True, debug=False)
except Exception as e:
    print("Không tạo được link share, chuyển sang link local:", e)
    demo.launch(share=False, debug=False)

print("Dùng xong có thể đóng tab này lại. Muốn mở lại chỉ cần chạy lại cell này.")


In [ ]:
# @title DÙNG TRỰC TIẾP BẰNG HÀM (tùy chọn)
def clone_voice(ref_file, gen_text, ref_text="", output="/content/output/result.wav",
                speed=1.0, seed=None, max_chars=100, nfe=32, cfg=2.0):
    """Nhân bản giọng ổn định: ref_file = file giọng mẫu, gen_text = văn bản cần đọc."""
    eng = load_engine(DEFAULT_MODEL)
    wave, sr, spec, seed_used = stable_generate(eng, ref_file, ref_text, gen_text,
                                                max_chars=max_chars, speed=speed,
                                                nfe=nfe, cfg=cfg, seed=seed)
    path = save_wav(wave, sr, output, remove_silence=True)
    return path, sr, seed_used

# Ví dụ cách dùng (bỏ comment để chạy):
# out, sr, seed_used = clone_voice("/content/my_voice.wav", "Xin chào, đây là giọng nói được nhân bản.")
# from IPython.display import Audio
# Audio(out)


## CÁCH TẢI KẾT QUẢ VỀ MÁY
* Trong giao diện, bấm biểu tượng tải xuống (download) của ô audio kết quả.
* Hoặc vào thư mục `/content/output/` trong Files (panel bên trái) và tải xuống.

## MẸO CHẤT LƯỢNG
* File giọng mẫu nên dài **5–12 giây**, rõ ràng, không nhạc nền, không ồn.
* **Nhập chính xác "Nội dung trong file giọng mẫu"** — nếu để trống máy phải tự nhận dạng bằng Whisper (chậm hơn và có thể sai, gây thiếu chữ).
* Bị **thiếu chữ**: giảm "Độ dài đoạn (max_chars)".
* Bị **vấp / lắp**: tăng nfe_step lên 48–64, hoặc tăng nhẹ max_chars lên 120–150.
* Với văn bản rất dài, máy tự chia nhỏ và ghép nối — cứ nhập thoải mái.

## GIỌNG ĐÃ LƯU
* Giọng đã lưu nằm trong thư mục `/content/voices/` (hoặc `MyDrive/VOICE_CLONE_TTS/voices/` nếu đã gắn Drive) — tải thư mục về máy để sao lưu.

## LIÊN HỆ MÃ NGUỒN
* F5-TTS: https://github.com/SWivid/F5-TTS
* Model tiếng Việt: https://huggingface.co/toandev/F5-TTS-Vietnamese
